# Three-Stage Ensemble (OCSVM) — Submission Generator

This notebook produces **`submission_blend_3Stage_OCSVM.npz`** only.

Pipeline: Stage 1 (LR+LGBM stack) → Stage 2 (blended unsupervised score) → Stage 3 (One-Class SVM novelty detector) → meta-LR combining all three signals.

## 1. Imports & Data Loading

In [14]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, roc_curve, precision_recall_curve,
    average_precision_score, classification_report
)
from sklearn.ensemble import (
    GradientBoostingClassifier, IsolationForest,
    StackingClassifier, VotingClassifier
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import LedoitWolf
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.base import clone as _clone2
from sklearn.ensemble import RandomForestClassifier

import os

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
    print("LightGBM available ✓")
except ImportError:
    print("LightGBM not installed — run: pip install lightgbm")
    LGBM_AVAILABLE = False

# ── Load training data ────────────────────────────────────────────────────────
data_folder = '/kaggle/input/datasets/victoriaquek/pml-data'

data_1 = np.load(os.path.join(data_folder, 'first_batch_with_labels.npz'))
data_2 = np.load(os.path.join(data_folder, 'second_batch_with_labels.npz'))
data_3 = np.load(os.path.join(data_folder, 'training_batch_with_labels.npz'))

# Combine all three labelled batches
X_raw = np.concatenate([data_1['X'], data_2['X'], data_3['X']], axis=0)
y_raw = np.concatenate([data_1['y'], data_2['y'], data_3['y']], axis=0)

print(f"Combined training interactions : {X_raw.shape[0]:,}")
print(f"Combined users                 : {len(np.unique(X_raw[:, 0]))}")
print(f"Anomalous users                : {np.sum(y_raw[:, 1] == 1)}")
print(f"Normal users                   : {np.sum(y_raw[:, 1] == 0)}")


LightGBM available ✓
Combined training interactions : 479,433
Combined users                 : 3060
Anomalous users                : 260
Normal users                   : 2800


## 2. Feature Engineering

In [2]:
# ── Shared handcrafted-feature function ──────────────────────────────────────
def compute_handcrafted_features(df, recon_error=None):
    """
    Per-user behavioural features from [user, item, rating] interactions.
    Appends recon_error as the last column if provided.
    """
    item_means      = df.groupby('item')['rating'].mean()
    df = df.copy()
    df['deviation'] = df['rating'] - df['item'].map(item_means)

    agg = df.groupby('user').agg(
        num_ratings     = ('rating', 'count'),
        mean_rating     = ('rating', 'mean'),
        std_rating      = ('rating', 'std'),
        num_items       = ('item',   'nunique'),
        pct_extreme     = ('rating', lambda r: ((r == 0) | (r == 5)).mean()),
        pct_zero        = ('rating', lambda r: (r == 0).mean()),
        pct_five        = ('rating', lambda r: (r == 5).mean()),
        rating_skewness = ('rating', lambda r: r.skew()),
        rating_kurtosis = ('rating', lambda r: r.kurtosis()),
        median_rating   = ('rating', 'median'),
        item_entropy    = ('item',   lambda x: -(
            pd.Series(x.value_counts(normalize=True)) *
            np.log2(pd.Series(x.value_counts(normalize=True)) + 1e-9)
        ).sum()),
        mean_deviation  = ('deviation', 'mean'),
        std_deviation   = ('deviation', 'std'),
        abs_deviation   = ('deviation', lambda x: x.abs().mean()),
        gini_items      = ('item', lambda x: 1 - sum(
            pd.Series(x).value_counts(normalize=True)**2)),
        pct_low         = ('rating', lambda r: (r < 3).mean()),
        # ── NEW: burst behaviour (anomalies often rate in tight time windows) ─
        # Approximated via rating concentration on a small item subset
        top5_item_pct   = ('item',   lambda x: x.value_counts(normalize=True).head(5).sum()),
        rating_range    = ('rating', lambda r: r.max() - r.min()),
    ).fillna(0)

    if recon_error is not None:
        agg['recon_error'] = recon_error.reindex(agg.index).fillna(0)

    return agg


# ── Build user-item matrix & labels ──────────────────────────────────────────
interactions = pd.DataFrame(X_raw, columns=['user', 'item', 'rating'])
labels_df    = pd.DataFrame(y_raw, columns=['user', 'label'])
labels_df    = labels_df.drop_duplicates(subset='user', keep='last')

user_item = interactions.pivot_table(
    index='user', columns='item', values='rating', aggfunc='mean'
).fillna(0)

labels_df = labels_df.set_index('user').loc[user_item.index].reset_index()
y = labels_df['label'].values

print(f"User-item matrix : {user_item.shape}")
print(f"Labels           : {y.shape}  |  balance {np.bincount(y)}")

# ── SVD reconstruction error ─────────────────────────────────────────────────
N_RECON = 30
_svd_recon = TruncatedSVD(n_components=N_RECON, random_state=42)
_ui         = user_item.values
_approx     = _svd_recon.fit_transform(_ui) @ _svd_recon.components_
recon_error = pd.Series(
    np.mean((_ui - _approx) ** 2, axis=1),
    index=user_item.index, name='recon_error'
)
print(f"Recon error — normal: {recon_error[y==0].mean():.4f}  "
      f"anomaly: {recon_error[y==1].mean():.4f}")

# ── Handcrafted features ──────────────────────────────────────────────────────
agg = compute_handcrafted_features(interactions, recon_error=recon_error)
agg = agg.loc[user_item.index]

# ── Cosine similarity to mean-normal user ────────────────────────────────────
normal_mask     = (y == 0)
mean_normal_vec = user_item.values[normal_mask].mean(axis=0, keepdims=True)
cos_sim         = cosine_similarity(user_item.values, mean_normal_vec).ravel()
agg['cosine_sim_to_normal'] = pd.Series(cos_sim, index=user_item.index)

# ── Mahalanobis distance ──────────────────────────────────────────────────────
lw = LedoitWolf().fit(agg.values[normal_mask])
mahal = np.sqrt(lw.mahalanobis(agg.values))
agg['mahal_dist'] = pd.Series(mahal, index=agg.index)

N_HANDCRAFTED = agg.shape[1]

# ── Raw combined feature matrix ───────────────────────────────────────────────
X_base = np.hstack([user_item.values, agg.values])
print(f"Feature matrix : {X_base.shape}  "
      f"({user_item.shape[1]} user-item + {N_HANDCRAFTED} handcrafted)")


User-item matrix : (3060, 998)
Labels           : (3060,)  |  balance [2800  260]
Recon error — normal: 0.7511  anomaly: 0.8854
Feature matrix : (3060, 1019)  (998 user-item + 21 handcrafted)


## 3. Custom Transformers

In [3]:
# ── SVDPlusHandcrafted: compress user-item block, pass through handcrafted ───
class SVDPlusHandcrafted(BaseEstimator, TransformerMixin):
    """TruncatedSVD on the user-item block; handcrafted columns pass through."""
    def __init__(self, n_components=50, n_handcrafted=None):
        self.n_components  = n_components
        self.n_handcrafted = n_handcrafted   # set at construction; None → use global

    def _nhc(self):
        return N_HANDCRAFTED if self.n_handcrafted is None else self.n_handcrafted

    def fit(self, X, y=None):
        self.svd_ = TruncatedSVD(n_components=self.n_components, random_state=42)
        self.svd_.fit(X[:, :-self._nhc()])
        return self

    def transform(self, X):
        svd_part  = self.svd_.transform(X[:, :-self._nhc()])
        hand_part = X[:, -self._nhc():]
        return np.hstack([svd_part, hand_part])


print("Custom transformers defined ✓")


Custom transformers defined ✓


## 4. Unsupervised Scores & Feature Matrix

In [4]:
# ── LOF: fit on NORMAL users only ────────────────────────────────────────────
print("Fitting LOF (normal users only) ...")
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.09,
    novelty=True,
    n_jobs=-1,
)
lof.fit(X_base[normal_mask])
lof_scores = -lof.score_samples(X_base)   # negate: higher = more anomalous

print(f"  LOF  — normal: {lof_scores[normal_mask].mean():.4f}  "
      f"anomaly: {lof_scores[~normal_mask].mean():.4f}")

# ── Isolation Forest: fit on NORMAL users only ───────────────────────────────
print("Fitting Isolation Forest (normal users only) ...")
iso = IsolationForest(
    n_estimators=300,
    contamination=0.09,
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_base[normal_mask])
iso_scores = -iso.score_samples(X_base)   # negate: higher = more anomalous

print(f"  ISO  — normal: {iso_scores[normal_mask].mean():.4f}  "
      f"anomaly: {iso_scores[~normal_mask].mean():.4f}")

# ── Normalise each signal to [0,1] using normal-user percentiles ──────────────
# def percentile_normalise(scores, ref_mask):
#     """Scale scores so the 1st–99th percentile of ref_mask spans [0, 1]."""
#     lo  = np.percentile(scores[ref_mask], 1)
#     hi  = np.percentile(scores[ref_mask], 99)
#     return np.clip((scores - lo) / (hi - lo + 1e-9), 0, None)

def percentile_normalise(scores, ref_mask=None, limits=None):
    """
    Scale scores so the 1st–99th percentile of ref_mask spans [0, 1].
    If limits are provided, use them directly (for test data).
    """
    if limits is not None:
        lo, hi = limits
    else:
        if ref_mask is None:
            ref_mask = np.ones(len(scores), dtype=bool)
        lo  = np.percentile(scores[ref_mask], 1)
        hi  = np.percentile(scores[ref_mask], 99)
    
    return np.clip((scores - lo) / (hi - lo + 1e-9), 0, None)

lof_norm   = percentile_normalise(lof_scores,   normal_mask)
iso_norm   = percentile_normalise(iso_scores,   normal_mask)
recon_norm = percentile_normalise(recon_error.values, normal_mask)

# ── Blend the three signals ───────────────────────────────────────────────────
# Equal weights; the meta-learner will learn the optimal combination downstream
unsup_score = (lof_norm + iso_norm + recon_norm) / 3.0

# AUC of the blended unsupervised signal alone
unsup_auc = roc_auc_score(y, unsup_score)
print(f"\nBlended unsupervised AUC : {unsup_auc:.4f}")

# ── Append to feature matrix ──────────────────────────────────────────────────
# X_with_unsup has three extra columns: lof_norm, iso_norm, unsup_score
X_with_unsup = np.hstack([
    X_base,
    lof_norm.reshape(-1, 1),
    iso_norm.reshape(-1, 1),
    unsup_score.reshape(-1, 1),
])
N_UNSUP_COLS   = 3
N_HC_PLUS_UNSUP = N_HANDCRAFTED + N_UNSUP_COLS
print(f"Extended feature matrix : {X_with_unsup.shape}")


Fitting LOF (normal users only) ...
  LOF  — normal: 1.0594  anomaly: 1.0697
Fitting Isolation Forest (normal users only) ...
  ISO  — normal: 0.4118  anomaly: 0.4044

Blended unsupervised AUC : 0.5239
Extended feature matrix : (3060, 1022)


## 5. Stage 1 — Supervised Stack

In [5]:
# ── SVD transformer that skips N_HC_PLUS_UNSUP trailing columns ─────────────
class SVDPlusAll(BaseEstimator, TransformerMixin):
    """SVD on user-item block; all trailing columns (handcrafted + unsup) pass through."""
    def __init__(self, n_components=80):
        self.n_components = n_components

    def fit(self, X, y=None):
        self.svd_ = TruncatedSVD(n_components=self.n_components, random_state=42)
        self.svd_.fit(X[:, :-N_HC_PLUS_UNSUP])
        return self

    def transform(self, X):
        svd_part  = self.svd_.transform(X[:, :-N_HC_PLUS_UNSUP])
        hand_part = X[:, -N_HC_PLUS_UNSUP:]
        return np.hstack([svd_part, hand_part])


class ColumnSelector(BaseEstimator, TransformerMixin):
    """Slice columns [:end_col] from the input matrix."""
    def __init__(self, end_col=None):
        self.end_col = end_col
    def fit(self, X, y=None): return self
    def transform(self, X):
        return X if self.end_col is None else X[:, :self.end_col]


# ── LR arm — receives X_base (no unsupervised columns) ───────────────────────
lr_pipe = Pipeline([
    ('svd', SVDPlusHandcrafted(n_components=80)),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced', C=0.1,
        max_iter=1000, solver='lbfgs', random_state=42,
    )),
])
lr_arm = Pipeline([
    ('cols',  ColumnSelector(end_col=X_base.shape[1])),   # drop unsup columns
    ('model', CalibratedClassifierCV(lr_pipe, method='sigmoid', cv=3)),
])

# ── LGBM arm — receives full X_with_unsup ────────────────────────────────────
if LGBM_AVAILABLE:
    tree_clf = LGBMClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.8,
        is_unbalance=True,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_samples=10,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    print("LGBM arm: LightGBMClassifier")
else:
    tree_clf = GradientBoostingClassifier(
        n_estimators=300, max_depth=4,
        learning_rate=0.03, subsample=0.8,
        min_samples_leaf=5, random_state=42,
    )
    print("LGBM arm: GradientBoostingClassifier (fallback)")

lgbm_pipe = Pipeline([
    ('svd',    SVDPlusAll(n_components=80)),
    ('scaler', StandardScaler()),
    ('clf',    tree_clf),
])
lgbm_arm = Pipeline([
    ('cols',  ColumnSelector(end_col=None)),   # full matrix
    ('model', lgbm_pipe),
])

# ── Stage 1 stacking classifier ───────────────────────────────────────────────
stage1_stack = StackingClassifier(
    estimators=[
        ('lr',   lr_arm),
        ('lgbm', lgbm_arm),
    ],
    final_estimator=LogisticRegression(
        C=0.5, class_weight='balanced',
        max_iter=1000, solver='lbfgs', random_state=42,
    ),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=1,
)

print("Stage 1 stacking classifier defined ✓")
print(f"  LR arm input  : {X_base.shape[1]} cols (user-item + handcrafted)")
print(f"  LGBM arm input: {X_with_unsup.shape[1]} cols (+ unsupervised scores)")


LGBM arm: LightGBMClassifier
Stage 1 stacking classifier defined ✓
  LR arm input  : 1019 cols (user-item + handcrafted)
  LGBM arm input: 1022 cols (+ unsupervised scores)


## 6. Two-Stage Ensemble Wrapper

In [6]:
# ── Two-stage meta ensemble wrapper ──────────────────────────────────────────
# class TwoStageEnsemble(BaseEstimator):
#     """
#     Wraps Stage1 (supervised stack) and Stage2 (blended unsupervised score).
#     The meta-LR combines their outputs.  Designed to work with cross_val_predict.

#     Parameters
#     ----------
#     stage1      : fitted or unfitted supervised pipeline/classifier
#     meta_C      : regularisation for meta logistic regression
#     unsup_score : per-user unsupervised score array (same row order as X)
#                   This is passed in via the constructor so cross_val_predict
#                   can slice the correct rows for each fold.
#     """
#     def __init__(self, stage1, meta_C=1.0, unsup_col_idx=-1):
#         self.stage1      = stage1
#         self.meta_C      = meta_C
#         self.unsup_col_idx = unsup_col_idx   # column index of unsup_score in X

#     def fit(self, X, y):
#         from sklearn.base import clone
#         from sklearn.model_selection import StratifiedKFold
#         import numpy as np

#         inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
#         s1_proba_oof = np.zeros(len(y))

#         stage1_clone = clone(self.stage1)
#         for tr, va in inner_cv.split(X, y):
#             stage1_clone_fold = clone(self.stage1)
#             stage1_clone_fold.fit(X[tr], y[tr])
#             s1_proba_oof[va] = stage1_clone_fold.predict_proba(X[va])[:, 1]

#         unsup = X[:, self.unsup_col_idx]
#         meta_X = np.column_stack([s1_proba_oof, unsup])

#         self.meta_ = LogisticRegression(
#             C=self.meta_C, class_weight='balanced',
#             max_iter=1000, solver='lbfgs', random_state=42
#         )
#         self.meta_.fit(meta_X, y)
#         self.meta_coef_ = self.meta_.coef_[0]

#         # Refit Stage 1 on the full fold data
#         self.stage1_fitted_ = clone(self.stage1)
#         self.stage1_fitted_.fit(X, y)
#         return self

#     def predict_proba(self, X):
#         s1_prob = self.stage1_fitted_.predict_proba(X)[:, 1]
#         unsup   = X[:, self.unsup_col_idx]
#         meta_X  = np.column_stack([s1_prob, unsup])
#         return self.meta_.predict_proba(meta_X)

#     def predict(self, X):
#         return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import numpy as np

# ── Two-stage meta ensemble wrapper ──────────────────────────────────────────
class TwoStageEnsemble(BaseEstimator, ClassifierMixin):
    """
    Wraps Stage1 (supervised stack) and Stage2 (blended unsupervised score).
    The meta-LR combines their outputs.  Designed to work with cross_val_predict.
    """
    def __init__(self, stage1, meta_C=1.0, unsup_col_idx=-1):
        self.stage1       = stage1
        self.meta_C       = meta_C
        self.unsup_col_idx = unsup_col_idx   # column index of unsup_score in X

    def fit(self, X, y):
        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
        s1_proba_oof = np.zeros(len(y))

        # Generate Stage 1 Out-of-Fold predictions for the meta-features
        for tr, va in inner_cv.split(X, y):
            stage1_clone_fold = clone(self.stage1)
            stage1_clone_fold.fit(X[tr], y[tr])
            s1_proba_oof[va] = stage1_clone_fold.predict_proba(X[va])[:, 1]

        unsup = X[:, self.unsup_col_idx]
        meta_X = np.column_stack([s1_proba_oof, unsup])

        self.meta_ = LogisticRegression(
            C=self.meta_C, class_weight='balanced',
            max_iter=1000, solver='lbfgs', random_state=42
        )
        self.meta_.fit(meta_X, y)
        
        # --- FIX: Set the classes_ attribute required by sklearn ---
        self.classes_ = self.meta_.classes_ 
        # -----------------------------------------------------------
        
        self.meta_coef_ = self.meta_.coef_[0]

        # Refit Stage 1 on the full fold data for final prediction
        self.stage1_fitted_ = clone(self.stage1)
        self.stage1_fitted_.fit(X, y)
        return self

    def predict_proba(self, X):
        s1_prob = self.stage1_fitted_.predict_proba(X)[:, 1]
        unsup   = X[:, self.unsup_col_idx]
        meta_X  = np.column_stack([s1_prob, unsup])
        return self.meta_.predict_proba(meta_X)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ── Unsupervised score is the last column of X_with_unsup ────────────────────
UNSUP_COL_IDX = -1   # last column = blended unsup_score

two_stage = TwoStageEnsemble(
    stage1=stage1_stack,
    meta_C=1.0,
    unsup_col_idx=UNSUP_COL_IDX,
)

# ── Cross-validate the full two-stage ensemble ────────────────────────────────
print("Cross-validating Two-Stage Ensemble ...")
print("(Each outer fold trains an inner 5-fold to generate Stage 1 OOF scores)")
print("This will take several minutes — each fold trains 2 x LR + 2 x LGBM stacks.")

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_scores_two_stage = cross_val_predict(
    two_stage, X_with_unsup, y,
    cv=outer_cv,
    method='predict_proba',
)[:, 1]

# ── Find F1-optimal threshold ─────────────────────────────────────────────────
_prec, _rec, _thresh = precision_recall_curve(y, y_scores_two_stage)
_f1 = np.where(
    (_prec[:-1] + _rec[:-1]) > 0,
    2 * _prec[:-1] * _rec[:-1] / (_prec[:-1] + _rec[:-1]),
    0
)
best_thresh_ts = float(_thresh[np.argmax(_f1)])
y_pred_ts      = (y_scores_two_stage >= best_thresh_ts).astype(int)

print("\n=== Two-Stage Ensemble (cross-validated) ===")
print(f"AUC       : {roc_auc_score(y, y_scores_two_stage):.4f}")
print(f"F1 thresh : {best_thresh_ts:.4f}")
print(f"Precision : {precision_score(y, y_pred_ts):.4f}")
print(f"Recall    : {recall_score(y, y_pred_ts):.4f}")
print(f"F1        : {f1_score(y, y_pred_ts):.4f}")
print()
print(classification_report(y, y_pred_ts, target_names=['Normal', 'Anomaly']))


Cross-validating Two-Stage Ensemble ...
(Each outer fold trains an inner 5-fold to generate Stage 1 OOF scores)
This will take several minutes — each fold trains 2 x LR + 2 x LGBM stacks.

=== Two-Stage Ensemble (cross-validated) ===
AUC       : 0.9505
F1 thresh : 0.8965
Precision : 0.7308
Recall    : 0.7308
F1        : 0.7308

              precision    recall  f1-score   support

      Normal       0.97      0.97      0.97      2800
     Anomaly       0.73      0.73      0.73       260

    accuracy                           0.95      3060
   macro avg       0.85      0.85      0.85      3060
weighted avg       0.95      0.95      0.95      3060



## 7. Helpers & Fit Final Two-Stage Model

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Helpers shared by all individual-submission cells
# ══════════════════════════════════════════════════════════════════════════════

def best_f1_threshold(y_true, y_scores):
    """Return the probability threshold that maximises F1 on the training set."""
    p, r, t = precision_recall_curve(y_true, y_scores)
    f1 = np.where((p[:-1]+r[:-1])>0, 2*p[:-1]*r[:-1]/(p[:-1]+r[:-1]), 0)
    return float(t[np.argmax(f1)])

def cv_report(name, y_true, y_scores):
    """Print AUC / F1 / P / R for a set of cross-validated scores."""
    thresh = best_f1_threshold(y_true, y_scores)
    preds  = (y_scores >= thresh).astype(int)
    print(f"{name:<38} AUC={roc_auc_score(y_true, y_scores):.4f}  "
          f"F1={f1_score(y_true, preds):.4f}  "
          f"P={precision_score(y_true, preds):.4f}  "
          f"R={recall_score(y_true, preds):.4f}")

# ── Fit the two-stage final model on all training data ───────────────────────
# (needed for Stage-1 scores on the test set)
print("Fitting final TwoStageEnsemble on all training data ...")
two_stage_final = TwoStageEnsemble(
    stage1=stage1_stack,
    meta_C=1.0,
    unsup_col_idx=UNSUP_COL_IDX,
)
two_stage_final.fit(X_with_unsup, y)
print("Done.")

# Also fit standalone baseline LR (no LGBM) for LR_* submissions
from sklearn.base import clone as _clone
baseline_lr_pipe = Pipeline([
    ('svd',    SVDPlusHandcrafted(n_components=80)),
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(class_weight='balanced', C=0.1,
                                   max_iter=1000, solver='lbfgs', random_state=42)),
])
baseline_lr_pipe.fit(X_base, y)
print("Baseline LR fitted.")


Fitting final TwoStageEnsemble on all training data ...
Done.
Baseline LR fitted.


## 8. Build Test Features

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# build_test_features — converts any raw interaction batch into the same
# X_with_unsup layout produced during training.
#
# Dependencies (must already be in scope, set during training):
#   _svd_recon, lw, lof, iso, mean_normal_vec
#   percentile_normalise, compute_handcrafted_features, cosine_similarity
# ══════════════════════════════════════════════════════════════════════════════

def build_test_features(X_interactions, all_items, svd_recon,
                        mean_normal_vec, lw_model, lof_model, iso_model,
                        lof_limits=None, iso_limits=None, recon_limits=None):
    """
    Build the combined feature matrix for a new batch of interactions.

    Parameters
    ----------
    X_interactions : np.ndarray, shape (n_interactions, 3)
        Columns: [user_id, item_id, rating]
    all_items : list
        Ordered item ids from training user-item pivot (user_item.columns.tolist()).
        Ensures column alignment with the training feature matrix.
    svd_recon : fitted TruncatedSVD
        Reconstruction-error SVD fitted on training data (_svd_recon).
    mean_normal_vec : np.ndarray, shape (1, n_items)
        Mean rating vector of normal training users.
    lw_model : fitted LedoitWolf
        Covariance model fitted on handcrafted features of normal training users.
    lof_model : fitted LocalOutlierFactor (novelty=True)
        LOF fitted on X_base of normal training users.
    iso_model : fitted IsolationForest
        Isolation Forest fitted on X_base of normal training users.
    lof_limits : tuple (lo, hi) or None
        Training-set percentile normalisation limits for LOF scores.
        Pass None to derive limits from this batch (risks test-set leakage).
    iso_limits : tuple (lo, hi) or None
        Training-set percentile normalisation limits for ISO scores.
    recon_limits : tuple (lo, hi) or None
        Training-set percentile normalisation limits for reconstruction error.

    Returns
    -------
    X_out : np.ndarray
        Shape (n_users, n_items + N_HANDCRAFTED + 3).
        Column layout mirrors training X_with_unsup:
          [user-item pivot | handcrafted features | lof_norm | iso_norm | unsup_blend]
    users : list
        User ids in the same row order as X_out.
    """
    df = pd.DataFrame(X_interactions, columns=["user", "item", "rating"])

    # ── User-item pivot — aligned to training item vocabulary ─────────────────
    pivot = (
        df.pivot_table(index="user", columns="item", values="rating", aggfunc="mean")
          .reindex(columns=all_items, fill_value=0)
          .fillna(0)
    )

    # ── SVD reconstruction error ──────────────────────────────────────────────
    _approx = svd_recon.transform(pivot.values) @ svd_recon.components_
    recon_e = pd.Series(
        np.mean((pivot.values - _approx) ** 2, axis=1),
        index=pivot.index,
        name="recon_error",
    )

    # ── Handcrafted features (same function as training) ──────────────────────
    agg_new = compute_handcrafted_features(df, recon_error=recon_e).loc[pivot.index]

    # ── Cosine similarity to mean normal training user ────────────────────────
    cos_sim_new = cosine_similarity(pivot.values, mean_normal_vec).ravel()
    agg_new["cosine_sim_to_normal"] = pd.Series(cos_sim_new, index=pivot.index)

    # ── Mahalanobis distance from normal-user centroid ────────────────────────
    mahal_new = np.sqrt(lw_model.mahalanobis(agg_new.values))
    agg_new["mahal_dist"] = mahal_new

    # ── Base feature matrix (user-item + handcrafted) ─────────────────────────
    X_b = np.hstack([pivot.values, agg_new.values])

    # ── Unsupervised scores ───────────────────────────────────────────────────
    lof_s = -lof_model.score_samples(X_b)   # negate → higher = more anomalous
    iso_s = -iso_model.score_samples(X_b)

    lof_n = percentile_normalise(lof_s,          limits=lof_limits)
    iso_n = percentile_normalise(iso_s,          limits=iso_limits)
    rec_n = percentile_normalise(recon_e.values, limits=recon_limits)

    unsup = (lof_n + iso_n + rec_n) / 3.0

    # ── Assemble: same column layout as X_with_unsup ──────────────────────────
    X_out = np.hstack([
        X_b,
        lof_n.reshape(-1, 1),
        iso_n.reshape(-1, 1),
        unsup.reshape(-1, 1),
    ])
    return X_out, pivot.index.tolist()


print("build_test_features defined ✓")
print("  Output layout: [user-item pivot | handcrafted | lof_norm | iso_norm | unsup_blend]")


build_test_features defined ✓
  Output layout: [user-item pivot | handcrafted | lof_norm | iso_norm | unsup_blend]


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Build test features (shared by all submission cells below)
# ══════════════════════════════════════════════════════════════════════════════

# Pre-compute normalisation limits from training normal users
lof_s_train  = -lof.score_samples(X_base)
iso_s_train  = -iso.score_samples(X_base)
recon_s_train = recon_error.values

lof_limits   = (np.percentile(lof_s_train[normal_mask],   1),
                np.percentile(lof_s_train[normal_mask],   99))
iso_limits   = (np.percentile(iso_s_train[normal_mask],   1),
                np.percentile(iso_s_train[normal_mask],   99))
recon_limits = (np.percentile(recon_s_train[normal_mask], 1),
                np.percentile(recon_s_train[normal_mask], 99))

all_items_list = user_item.columns.tolist()

# Load test batch
data_test  = np.load(os.path.join(data_folder, 'third_batch.npz'))
X_test_raw = data_test['X']

# Build full test feature matrix (includes unsupervised columns)
X_test_with_unsup, test_users = build_test_features(
    X_test_raw,
    all_items       = all_items_list,
    svd_recon       = _svd_recon,
    mean_normal_vec = mean_normal_vec,
    lw_model        = lw,
    lof_model       = lof,
    iso_model       = iso,
    lof_limits      = lof_limits,
    iso_limits      = iso_limits,
    recon_limits    = recon_limits,
)
# X_base slice (no unsupervised cols) for the LR-only submissions
X_test_base = X_test_with_unsup[:, :X_base.shape[1]]

# Extract individual normalised unsupervised test scores
# Column layout of X_with_unsup: [...X_base... | lof_norm | iso_norm | unsup_blend]
lof_col_idx  = X_base.shape[1]          # lof_norm column
iso_col_idx  = X_base.shape[1] + 1      # iso_norm column
unsup_col_idx_test = X_base.shape[1] + 2   # blended unsup score column

test_lof_norm   = X_test_with_unsup[:, lof_col_idx]
test_iso_norm   = X_test_with_unsup[:, iso_col_idx]
test_recon_norm = percentile_normalise(
    -_svd_recon.transform(X_test_base[:, :user_item.shape[1]]) @ _svd_recon.components_,
    limits=recon_limits,
)
# Recompute recon for test properly
df_test_tmp = pd.DataFrame(X_test_raw, columns=['user','item','rating'])
pivot_test_tmp = df_test_tmp.pivot_table(index='user',columns='item',
    values='rating',aggfunc='mean').reindex(columns=all_items_list,fill_value=0).fillna(0)
_approx_test = _svd_recon.transform(pivot_test_tmp.values) @ _svd_recon.components_
recon_test_raw = np.mean((pivot_test_tmp.values - _approx_test)**2, axis=1)
test_recon_norm = percentile_normalise(recon_test_raw, limits=recon_limits)

print(f"Test feature matrix shape: {X_test_with_unsup.shape}")
print(f"Test users: {len(test_users)}")


Test feature matrix shape: (1625, 1022)
Test users: 1625


## 9. Stage 3 — One-Class SVM & Three-Stage Ensemble

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Method D — Three-Stage Cascade (adds One-Class SVM novelty detector)
# ══════════════════════════════════════════════════════════════════════════════
#
# Stage 3 adds a One-Class SVM (OCSVM) trained on the HANDCRAFTED feature
# space of normal users.  OCSVM is sensitive to smooth manifold deviations
# that both IF and LOF can miss when the feature space is dense.
# The OCSVM score is appended and the meta-LR now combines 3 signals:
#   [stage1_prob, blended_unsup_score, ocsvm_score]
from sklearn.svm import OneClassSVM

print("Fitting One-Class SVM (normal users, handcrafted feature space) ...")
# Use handcrafted features only (compact, meaningful) for OCSVM
# agg.values = handcrafted features; already computed globally
hc_scaler = StandardScaler().fit(agg.values[normal_mask])
hc_normal  = hc_scaler.transform(agg.values[normal_mask])

ocsvm = OneClassSVM(kernel='rbf', nu=0.09, gamma='scale')
ocsvm.fit(hc_normal)

# Score: decision_function < 0 → anomaly; negate so higher = more anomalous
ocsvm_scores = -ocsvm.decision_function(hc_scaler.transform(agg.values))
ocsvm_norm   = percentile_normalise(ocsvm_scores, normal_mask)
print(f"  OCSVM — normal: {ocsvm_norm[normal_mask].mean():.4f}  "
      f"anomaly: {ocsvm_norm[~normal_mask].mean():.4f}")

# Append OCSVM column to feature matrix
X_with_ocsvm = np.hstack([X_with_unsup, ocsvm_norm.reshape(-1, 1)])
OCSVM_COL_IDX = -1   # last column


class ThreeStageEnsemble(BaseEstimator, ClassifierMixin):
    """
    Extends TwoStageEnsemble:  meta-LR receives [stage1_prob, unsup_score, ocsvm_score].
    """
    def __init__(self, stage1, meta_C=1.0, unsup_col=-2, ocsvm_col=-1):
        self.stage1    = stage1
        self.meta_C    = meta_C
        self.unsup_col = unsup_col
        self.ocsvm_col = ocsvm_col

    def fit(self, X, y):
        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
        s1_oof = np.zeros(len(y))
        for tr, va in inner_cv.split(X, y):
            c = _clone2(self.stage1); c.fit(X[tr], y[tr])
            s1_oof[va] = c.predict_proba(X[va])[:, 1]
        meta_X = np.column_stack([s1_oof,
                                  X[:, self.unsup_col],
                                  X[:, self.ocsvm_col]])
        self.meta_ = LogisticRegression(C=self.meta_C, class_weight='balanced',
                                         max_iter=1000, solver='lbfgs', random_state=42)
        self.meta_.fit(meta_X, y)
        self.classes_ = self.meta_.classes_
        self.meta_coef_ = self.meta_.coef_[0]
        self.stage1_ = _clone2(self.stage1); self.stage1_.fit(X, y)
        return self

    def predict_proba(self, X):
        s1p    = self.stage1_.predict_proba(X)[:, 1]
        meta_X = np.column_stack([s1p,
                                  X[:, self.unsup_col],
                                  X[:, self.ocsvm_col]])
        return self.meta_.predict_proba(meta_X)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


ts3 = ThreeStageEnsemble(
    stage1=stage1_stack, meta_C=1.0,
    unsup_col=-2,   # blended unsup score is now second-to-last
    ocsvm_col=-1,
)
y_cv_3stage = cross_val_predict(
    ts3, X_with_ocsvm, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    method='predict_proba',
)[:, 1]
cv_report('Method D: Three-Stage (+ OCSVM)', y, y_cv_3stage)


Fitting One-Class SVM (normal users, handcrafted feature space) ...
  OCSVM — normal: 0.1657  anomaly: 0.2677
Method D: Three-Stage (+ OCSVM)        AUC=0.9572  F1=0.7442  P=0.8263  R=0.6769


## 10. Fit Final Three-Stage Model on All Training Data

In [12]:
class TwoStageRFMeta(BaseEstimator, ClassifierMixin):
    """TwoStageEnsemble variant with RF meta-learner instead of LR."""
    def __init__(self, stage1, n_estimators=100, unsup_col_idx=-1):
        self.stage1        = stage1
        self.n_estimators  = n_estimators
        self.unsup_col_idx = unsup_col_idx

    def fit(self, X, y):
        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
        s1_oof = np.zeros(len(y))
        for tr, va in inner_cv.split(X, y):
            c = _clone2(self.stage1); c.fit(X[tr], y[tr])
            s1_oof[va] = c.predict_proba(X[va])[:, 1]
        unsup  = X[:, self.unsup_col_idx]
        meta_X = np.column_stack([s1_oof, unsup])
        self.meta_ = RandomForestClassifier(
            n_estimators=self.n_estimators,
            class_weight='balanced',
            max_depth=4,
            random_state=42,
            n_jobs=-1,
        )
        self.meta_.fit(meta_X, y)
        self.classes_ = self.meta_.classes_
        self.stage1_  = _clone2(self.stage1); self.stage1_.fit(X, y)
        return self

    def predict_proba(self, X):
        s1p    = self.stage1_.predict_proba(X)[:, 1]
        meta_X = np.column_stack([s1p, X[:, self.unsup_col_idx]])
        return self.meta_.predict_proba(meta_X)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# Fit final models on all training data for blending submissions
# ══════════════════════════════════════════════════════════════════════════════

print("Fitting final models on all training data ...")

# Stage 1 final (already fitted above as two_stage_final.stage1_)
# We'll use two_stage_final.stage1_ directly for all blend submissions.

# RF Meta-Learner final
ts_rf_final = TwoStageRFMeta(stage1=stage1_stack, n_estimators=200, unsup_col_idx=-1)
ts_rf_final.fit(X_with_unsup, y)

# Three-Stage + OCSVM final
ts3_final = ThreeStageEnsemble(stage1=stage1_stack, meta_C=1.0, unsup_col=-2, ocsvm_col=-1)
ts3_final.fit(X_with_ocsvm, y)

# ── OCSVM scores for test set ────────────────────────────────────────────────
# X_test_with_unsup column layout (set by build_test_features):
#   [user-item pivot (n_items) | handcrafted (N_HANDCRAFTED) | lof_norm | iso_norm | unsup_blend]
# hc_scaler was fitted on agg.values which has exactly N_HANDCRAFTED columns
# (including cosine_sim_to_normal and mahal_dist). Slicing directly from
# X_test_with_unsup avoids any column-count mismatch.
n_items_train   = user_item.shape[1]
hc_start        = n_items_train
hc_end          = n_items_train + N_HANDCRAFTED
agg_test_values = X_test_with_unsup[:, hc_start:hc_end]  # shape (n_test, N_HANDCRAFTED)

ocsvm_test_raw  = -ocsvm.decision_function(hc_scaler.transform(agg_test_values))

# Normalise using same limits derived from training normal users
_ocsvm_train_raw = -ocsvm.decision_function(hc_scaler.transform(agg.values[normal_mask]))
ocsvm_limits     = (np.percentile(_ocsvm_train_raw, 1),
                    np.percentile(_ocsvm_train_raw, 99))
ocsvm_test_norm  = percentile_normalise(ocsvm_test_raw, limits=ocsvm_limits)

X_test_with_ocsvm = np.hstack([X_test_with_unsup, ocsvm_test_norm.reshape(-1, 1)])
print("Final models fitted.")

Fitting final models on all training data ...
Final models fitted.


## 11. Save Submission

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# Save submission_blend_3Stage_OCSVM.npz
# ══════════════════════════════════════════════════════════════════════════════

def save_submission(filename, scores):
    np.savez(filename, predictions=scores)
    print(f"  Saved {filename}  (range [{scores.min():.4f}, {scores.max():.4f}], mean {scores.mean():.4f})")

print("Generating submission_blend_3Stage_OCSVM.npz ...")

# Method D: Three-Stage + OCSVM
blend_d_test = ts3_final.predict_proba(X_test_with_ocsvm)[:, 1]
save_submission('submission_blend_3Stage_OCSVM.npz', blend_d_test)

print("Done.")


Generating submission_blend_3Stage_OCSVM.npz ...
  Saved submission_blend_3Stage_OCSVM.npz  (range [0.0161, 0.9855], mean 0.1799)
Done.
